## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework --pre
pip install --upgrade azure-ai-projects --pre
```

## 📒 Notebook 4: Memory

### 🪜 Step 1: Configure Environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import json
import asyncio
from agent_framework.azure import AzureAIClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Fix: Disable Accept-Encoding to prevent compressed responses
import azure.core.pipeline.policies as policies

# Store original
_original_on_request = policies.HeadersPolicy.on_request

def patched_on_request(self, request):
    """Remove Accept-Encoding header to prevent compressed responses."""
    _original_on_request(self, request)
    # Tell server we don't accept compressed responses
    request.http_request.headers['Accept-Encoding'] = 'identity'

policies.HeadersPolicy.on_request = patched_on_request
print("Applied Accept-Encoding fix to disable compression")

Applied Accept-Encoding fix to disable compression


In [4]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Create AI Agent

In [5]:
# Define AI client
ai_client = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-Concierge",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: MAFSDK-Agentv2-Concierge


In [6]:
# Create Concierge agent
agent = ai_client.create_agent(
    name = "hotel-concierge-agent",
    instructions = (
        "You are a friendly and professional hotel concierge. "
        "Always remember the guest's name and room number after they check in."
    )
)

print(f"Created AI agent: {agent.name}")

Created AI agent: hotel-concierge-agent


### 🪜 Step 3: Run Agent and Serialise Thread

In [7]:
# Create thread for the guest conversation
guest_thread = agent.get_new_thread()

print(f"New guest thread created")

New guest thread created


In [8]:
# Simulate guest check-in
PROMPT1 = "Hello, my name is Alex Reed and I am checking into room 1205."
print(f"User (Check-in):\n{PROMPT1}\n")

response = await agent.run(PROMPT1, thread=guest_thread)
print(f"Agent:\n{response.text}\n")

User (Check-in):
Hello, my name is Alex Reed and I am checking into room 1205.

Agent:
Welcome, Alex Reed! I have you checked into room 1205. If you need any assistance during your stay, feel free to ask. How can I help you today?



In [9]:
# Serialise the thread state
serialised_thread_data = await guest_thread.serialize()

# Save thread state to a local JSON file
FILE_NAME = "concierge_thread_memory.json"
with open(FILE_NAME, "w") as f:
    json.dump(serialised_thread_data, f, indent=4)
    
print(f"Thread data successfully serialised and saved to {FILE_NAME}")

Thread data successfully serialised and saved to concierge_thread_memory.json


### 🪜 Step 4: Application Restart

In [10]:
# Close AI client and delete all objects from memory
await ai_client.close()
del agent
del ai_client
del guest_thread

print("Azure AI client, agent and thread objects cleared to simulate memory loss")

Azure AI client, agent and thread objects cleared to simulate memory loss


### 🪜 Step 5: Re-initialise Objects and Deserialise Thread

In [11]:
# Re-create the AI client and agent
ai_client_new = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-Concierge",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

agent_new = ai_client_new.create_agent(
    name = "hotel-concierge-agent",
    instructions = (
        "You are a friendly and professional hotel concierge. "
        "Always remember the guest's name and room number after they check in."
    )
)

print(f"New AI client and agent objects created after simulated memory loss")

New AI client and agent objects created after simulated memory loss


In [12]:
# Load and deserialise thread from JSON file
with open(FILE_NAME, "r") as f:
    thread_data_reloaded = json.load(f)
    
restored_thread = await agent_new.deserialize_thread(thread_data_reloaded)
print(f"Thread successfully deserialised. Type: {type(restored_thread)}")

Thread successfully deserialised. Type: <class 'agent_framework._threads.AgentThread'>


In [13]:
# Continue conversation using the restored thread
PROMPT2 = "I would like to order room service: a club sandwich and a pot of tea."
print(f"User (Room Service):\n{PROMPT2}\n")

response_final = await agent_new.run(PROMPT2, thread=restored_thread)
print(f"Agent:\n{response_final.text}")

User (Room Service):
I would like to order room service: a club sandwich and a pot of tea.

Agent:
Certainly, Alex! I’ll place an order for a club sandwich and a pot of tea to be delivered to room 1205. Is there any particular time you would like this to be served?


### 🪜 Step 6: Housekeeping

In [14]:
# Delete Azure AI client
await ai_client_new.close()

print("AI client closed successfully.")

AI client closed successfully.
